In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
spark.sql(F'USE CATALOG {catalog_name}')


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze;

###SQL Data Ingestion

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

CUSTOMERS_SCHEMA = StructType(
    [
        StructField("customer_id", IntegerType()),
        StructField("first_name", StringType()),
        StructField("last_name", StringType()),
        StructField("email", StringType()),
        StructField("phone", StringType()),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("country", StringType()),
        StructField("customer_status", StringType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)
 
ORDERS_SCHEMA = StructType(
    [
        StructField("order_id", IntegerType()),
        StructField("customer_id", IntegerType()),
        StructField("order_date", TimestampType()),
        StructField("order_status", StringType()),
        StructField("shipping_address", StringType()),
        StructField("total_amount", DecimalType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)

ORDER_ITEMS_SCHEMA = StructType(
    [
        StructField("order_item_id", LongType()),
        StructField("order_id", LongType()),
        StructField("product_id", IntegerType()),
        StructField("quantity",IntegerType()),
        StructField("unit_price", DecimalType()),
        StructField("discount", DecimalType()),
        StructField("created_at", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)

PAYMENTS_SCHEMA = StructType(
    [
        StructField("payment_id", LongType()),
        StructField("order_id", LongType()),
        StructField("payment_method", StringType()),
        StructField("payment_amount", DecimalType()),
        StructField("payment_status", StringType()),
        StructField("payment_date", TimestampType()),
        StructField("updated_at", TimestampType()),
    ]
)

PRODUCTS_SCHEMA = StructType(
    [
        StructField("product_id", IntegerType()),
        StructField("product_name",StringType()),
        StructField("category", StringType()),
        StructField("subcategory", StringType()),
        StructField("brand", StringType()),
        StructField("price", DecimalType()),
        StructField("cost", DecimalType()),
        StructField("supplier_id", StringType()),
        StructField("product_status", StringType()),
        StructField("updated_at", TimestampType()),
    ]
)

INVENTORY_SCHEMA = StructType(
    [
        StructField("inventory_date", DateType()),
        StructField("product_id", IntegerType()),
        StructField("warehouse_id", StringType()),
        StructField("available_quantity", IntegerType()),
        StructField("reserved_quantity", IntegerType()),
        StructField("reorder_level", IntegerType()),
    ]
)

CLICKSTREAM_SCHEMA = StructType(
    [
        StructField("event_id", LongType()),
        StructField("customer_id", IntegerType()),
        StructField("session_id", StringType()),
        StructField("event_time", TimestampType()),
        StructField("event_type", StringType()),
        StructField("product_id", IntegerType()),
        StructField("page", StringType()),
        StructField("device_type", StringType()),
    ]
)


In [0]:
customers_df = (spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMERS_SCHEMA)
    .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/customers/"))
 
customers_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/customers",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.customers"
)

In [0]:
orders_df = (spark.readStream
             .format("csv")
             .option("header", "true")
             .schema(ORDERS_SCHEMA)
             .load("/Volumes/shopsphere/ecomm_schema/capstone_data/sql_server/initial/orders/"))


 
orders_df.writeStream.option(
    "checkpointLocation",
    "/Volumes/shopsphere/ecomm_schema/capstone_data/checkpoint/orders",
).format('delta').trigger(availableNow=True).outputMode("Append").toTable(
    "shopsphere.bronze.orders"
)

In [0]:

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "shopsphere.bronze.customers")

delta_table.history().display()

###ADLS Data Ingestion

In [0]:
products_df = spark.read.csv(path='/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/products/', header=True, inferSchema=True)

inventory_df = spark.read.csv(path='/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/inventory/', header=True, inferSchema=True)

clickstream_df = spark.read.json(path='/Volumes/shopsphere/ecomm_schema/capstone_data/adls/initial/clickstream/')


In [0]:
products_df.write.saveAsTable('bronze.products',mode='OVERWRITE')
inventory_df.write.saveAsTable('bronze.inventory',mode='OVERWRITE')
clickstream_df.write.saveAsTable('bronze.clickstream',mode='OVERWRITE')